# Partie 3 — Pandas

**Objectif** : charger, explorer, nettoyer et analyser un vrai fichier de données.

Pandas, c'est **Excel en Python** : un tableau avec des lignes et des colonnes nommées, qu'on appelle un **DataFrame**. Chaque colonne s'appelle une **Series**.

**Les données du jour** : `data/ventes.csv` contient les ventes de 6 mois (janvier à juin 2026) d'une petite chaîne de coffee shops, avec 4 magasins (Paris, Lyon, Marseille, Lille). Les données sont fictives, et elles contiennent quelques erreurs, comme dans la vraie vie.

In [ ]:
import pandas as pd   # convention : on l'appelle toujours pd

## 1. Créer un DataFrame à la main

Un DataFrame peut se créer à partir d'un dictionnaire : chaque clé devient une colonne.

In [ ]:
mini = pd.DataFrame({
    "produit": ["Croissant", "Latte", "Cookie"],
    "prix": [1.3, 4.0, 2.5],
    "quantite": [40, 25, 12],
})
mini

In [ ]:
mini["prix"]          # une colonne = une Series


## 2. Charger et explorer un fichier

Les **5 réflexes** quand on ouvre un fichier de données :

| Commande | Ce qu'elle montre |
| --- | --- |
| `df.head()` | les 5 premières lignes |
| `df.shape` | (nombre de lignes, nombre de colonnes) |
| `df.info()` | le type de chaque colonne et les valeurs manquantes |
| `df.describe()` | les statistiques des colonnes numériques |
| `df["col"].value_counts()` | combien de fois chaque valeur apparaît |

In [ ]:
df = pd.read_csv("data/ventes.csv")
df.head()

In [ ]:
print(df.shape)
df.info()

In [ ]:
df.describe()

In [ ]:
df["magasin"].value_counts()

### Exercice 1 · Niveau 1 · facile · en binôme

1. Affichez les **10 premières lignes** (indice : `head(10)`).
2. Affichez les **5 dernières** (indice : `tail()`).
3. Combien y a-t-il de **catégories** différentes, et combien de ventes dans chacune ?

In [ ]:
# Votre code ici


## 3. Sélectionner et filtrer

- Une colonne : `df["magasin"]`
- Plusieurs colonnes : `df[["magasin", "produit"]]` (attention aux **doubles crochets**)
- Filtrer les lignes : `df[condition]`
- Plusieurs conditions : `&` (et), `|` (ou), avec des **parenthèses** autour de chaque condition.

In [ ]:
df[["date", "magasin", "produit"]].head()

In [ ]:
paris = df[df["magasin"] == "Paris"]
print(len(paris), "ventes à Paris")

grosses_ventes_lyon = df[(df["magasin"] == "Lyon") & (df["quantite"] >= 4)]
grosses_ventes_lyon.head()

In [ ]:
# .loc[lignes, colonnes] : sélectionner par conditions et noms de colonnes
df.loc[df["categorie"] == "Thé", ["date", "produit", "quantite"]].head()

### Exercice 2 · Niveau 1 · facile · en binôme

1. Gardez seulement les ventes du magasin de **Marseille**. Combien y en a-t-il ?
2. Gardez les ventes de la catégorie **Pâtisserie** avec une quantité **supérieure ou égale à 3**.

In [ ]:
# Votre code ici


## 4. Créer une colonne

On crée une colonne comme on remplit une case de dictionnaire. Le calcul se fait **sur toutes les lignes d'un coup**, comme avec NumPy.

In [ ]:
df["montant"] = df["quantite"] * df["prix_unitaire"]
df.head()

### Exercice 3 · Niveau 2 · moyen · en binôme

1. Quel est le **chiffre d'affaires total** (la somme de la colonne `montant`) ?
2. Quel est le **montant moyen** d'une vente ?
3. Quelle est la **plus grosse vente** ? Affichez toute la ligne (indice : `idxmax()` puis `df.loc[...]`).

In [ ]:
# Votre code ici


## 5. Nettoyer les données

Les données réelles sont rarement propres. Deux problèmes classiques :

- **Les valeurs manquantes** (`NaN`) : `df.isna().sum()` les compte, `dropna()` supprime les lignes concernées, `fillna(valeur)` les remplace.
- **Les doublons** : `df.duplicated().sum()` les compte, `drop_duplicates()` les supprime.

Remarquez : la colonne `montant` vaut `NaN` quand la quantité manque. Un calcul avec une valeur manquante donne une valeur manquante.

In [ ]:
df.isna().sum()

In [ ]:
df[df["quantite"].isna()].head()

### Exercice 4 · Niveau 2 · moyen · en binôme

1. Combien de lignes ont une **quantité manquante** ?
2. Combien de lignes sont des **doublons** exacts ?
3. Créez un DataFrame `propre` sans valeurs manquantes et sans doublons. Combien de lignes reste-t-il ?
4. Recalculez le CA total sur `propre`. Est-il différent ? Pourquoi ?

In [ ]:
# Votre code ici


**À partir d'ici, on travaille sur les données propres.**

In [ ]:
propre = df.dropna(subset=["quantite"]).drop_duplicates().copy()
propre["date"] = pd.to_datetime(propre["date"])   # transformer le texte en vraie date
propre.info()

## 6. Trier

In [ ]:
propre.sort_values("montant", ascending=False).head()   # les plus grosses ventes d'abord

## 7. Regrouper avec `groupby`

C'est **la** fonction la plus utile de Pandas. Elle répond aux questions du type « **quel est le total par magasin ?** ».

On la lit en trois temps : **regrouper** par une colonne, **choisir** la colonne à calculer, **appliquer** un calcul.

```python
propre.groupby("magasin")["montant"].sum()
#        regrouper       choisir    calculer
```

In [ ]:
propre.groupby("magasin")["montant"].sum().sort_values(ascending=False)

In [ ]:
# Plusieurs calculs d'un coup avec .agg
propre.groupby("magasin").agg(
    ca=("montant", "sum"),
    nb_ventes=("id_vente", "count"),
    panier_moyen=("montant", "mean"),
).round(2)

### Exercice 5 · Niveau 2 · moyen · en binôme

1. Calculez le **CA par catégorie**, du plus grand au plus petit.
2. Quel est le **produit le plus vendu en quantité** ?
3. Calculez le **CA par magasin et par catégorie** (indice : `groupby(["magasin", "categorie"])`).

In [ ]:
# Votre code ici


## 8. Travailler avec les dates

Une fois la colonne convertie avec `pd.to_datetime`, on accède au mois, au jour de la semaine, etc. avec `.dt`.

In [ ]:
propre["mois"] = propre["date"].dt.month
propre["jour_semaine"] = propre["date"].dt.day_name()
propre[["date", "mois", "jour_semaine"]].head()

### Exercice 6 · Niveau 3 · défi · en binôme

**Défi** :
1. Calculez le **CA par mois**. Quel est le meilleur mois ?
2. Les **boissons fraîches** se vendent-elles plus en été ? Calculez le CA de cette catégorie par mois.
3. Quel est le **jour de la semaine** où l'on vend le plus ?

In [ ]:
# Votre code ici


## 9. Fusionner deux tableaux avec `merge`

Le fichier `data/magasins.csv` contient des informations sur chaque magasin (région, surface). Pour les ajouter aux ventes, on **fusionne** les deux tableaux sur la colonne commune `magasin`. C'est l'équivalent de RECHERCHEV dans Excel.

In [ ]:
magasins = pd.read_csv("data/magasins.csv")
magasins

In [ ]:
ventes_completes = propre.merge(magasins, on="magasin", how="left")
ventes_completes[["date", "magasin", "region", "surface_m2", "montant"]].head()

### Exercice 7 · Niveau 3 · défi · en binôme

**Défi** : calculez le **CA par mètre carré** de chaque magasin (CA du magasin divisé par sa surface).
Le magasin qui vend le plus est-il aussi le plus rentable au m² ?

In [ ]:
# Votre code ici


## 10. Un premier graphique

Pandas sait dessiner directement avec `.plot()`. C'est la transition vers la dernière partie : la **visualisation** dans une application web.

In [ ]:
import matplotlib.pyplot as plt

propre.groupby("mois")["montant"].sum().plot(kind="line", marker="o", title="CA par mois (€)")
plt.show()

propre.groupby("magasin")["montant"].sum().sort_values().plot(kind="barh", title="CA par magasin (€)")
plt.show()

## Récapitulatif

| Je veux… | J'écris |
| --- | --- |
| charger un CSV | `pd.read_csv("fichier.csv")` |
| explorer | `head()`, `shape`, `info()`, `describe()`, `value_counts()` |
| filtrer | `df[df["col"] == valeur]` |
| créer une colonne | `df["c"] = df["a"] * df["b"]` |
| nettoyer | `isna().sum()`, `dropna()`, `drop_duplicates()` |
| trier | `sort_values("col", ascending=False)` |
| regrouper | `df.groupby("col")["autre"].sum()` |
| fusionner | `df.merge(autre, on="col")` |
| dessiner | `.plot(kind="line")` |